<a href="https://colab.research.google.com/github/lankipolo123/roadfixqc/blob/main/Final_Pothole_train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================================
# FIXED - Check folder structure first
# ============================================================================

from google.colab import drive
drive.mount('/content/drive')

print("✅ Google Drive mounted!")

# INSTALL
!pip install ultralytics roboflow

import os
import shutil
from ultralytics import YOLO
import torch
from google.colab import files

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# DOWNLOAD DATASET
from roboflow import Roboflow
rf = Roboflow(api_key="SzttdelfmuWaCwAz2N5u")
project = rf.workspace("dequillaprojects").project("pothole-ol3a7")
version = project.version(7)
dataset = version.download("yolov11")

dataset_path = dataset.location
print(f"Dataset downloaded to: {dataset_path}")

# ============================================================================
# FIX: CHECK ACTUAL FOLDER STRUCTURE
# ============================================================================

print("\n📁 Checking dataset structure...")
print("Contents of dataset folder:")
for item in os.listdir(dataset_path):
    print(f"  - {item}")

# Find the correct folder names
if os.path.exists(f"{dataset_path}/train/images"):
    train_folder = "train"
    val_folder = "valid"
elif os.path.exists(f"{dataset_path}/train"):
    train_folder = "train"
    val_folder = "val"  # Sometimes it's 'val' not 'valid'
else:
    # List all to see structure
    print("\n❌ Standard structure not found. Full structure:")
    for root, dirs, files in os.walk(dataset_path):
        level = root.replace(dataset_path, '').count(os.sep)
        indent = ' ' * 2 * level
        print(f'{indent}{os.path.basename(root)}/')
        if level < 2:  # Only show 2 levels deep
            subindent = ' ' * 2 * (level + 1)
            for file in files[:5]:  # Show first 5 files
                print(f'{subindent}{file}')

    raise Exception("Please check the folder structure above and update the script")

# Count images
train_images = len(os.listdir(f"{dataset_path}/{train_folder}/images"))
val_images = len(os.listdir(f"{dataset_path}/{val_folder}/images"))
print(f"\n✅ Training images: {train_images}")
print(f"✅ Validation images: {val_images}")

# ============================================================================
# REST OF TRAINING (SAME AS BEFORE)
# ============================================================================

print("\nTraining YOLOv11n - Distance + Blur Pothole Detection...")

drive_project_path = '/content/drive/MyDrive/pothole_distance_blur_training'
os.makedirs(drive_project_path, exist_ok=True)

model = YOLO('yolo11n.pt')

results = model.train(
    data=f'{dataset_path}/data.yaml',
    epochs=250,
    imgsz=832,
    batch=24,
    lr0=0.001,
    lrf=0.01,
    optimizer='AdamW',
    cos_lr=True,
    patience=60,
    save_period=10,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=10,
    translate=0.15,
    scale=0.5,
    shear=5,
    fliplr=0.5,
    flipud=0.0,
    mosaic=1.0,
    mixup=0.15,
    box=8.0,
    cls=0.5,
    dfl=1.5,
    name='yolov11n_pothole_distance_blur_v7',
    project=drive_project_path,
    exist_ok=True
)

print("✅ Training saved to Drive!")

val_results = model.val()
print(f"\nmAP50: {val_results.box.map50:.4f}")
print(f"mAP50-95: {val_results.box.map:.4f}")

if val_results.box.map50 < 0.90:
    print(f"\nCurrent: {val_results.box.map50*100:.1f}% - Fine-tuning...")

    best_model_path = f'{drive_project_path}/yolov11n_pothole_distance_blur_v7/weights/best.pt'
    fine_tune_model = YOLO(best_model_path)

    fine_tune_results = fine_tune_model.train(
        data=f'{dataset_path}/data.yaml',
        epochs=100,
        imgsz=1024,
        batch=12,
        lr0=0.0001,
        optimizer='AdamW',
        cos_lr=True,
        patience=30,
        mosaic=0.3,
        mixup=0.05,
        scale=0.5,
        box=8.0,
        name='yolov11n_finetune_distance_blur_v7',
        project=drive_project_path,
        exist_ok=True
    )

    val_results = fine_tune_model.val()
    print(f"Fine-tuned mAP50: {val_results.box.map50:.4f}")

final_accuracy = val_results.box.map50 * 100
print("\n" + "="*50)
print("DISTANCE + BLUR POTHOLE DETECTION - RESULTS")
print("="*50)
print(f"Accuracy: {final_accuracy:.1f}%")
print(f"mAP50-95: {val_results.box.map:.4f}")
print(f"Precision: {val_results.box.mp:.4f}")
print(f"Recall: {val_results.box.mr:.4f}")

final_model_name = f"BEST_Pothole_Distance_Blur_{final_accuracy:.1f}percent.pt"
best_path = f'{drive_project_path}/yolov11n_pothole_distance_blur_v7/weights/best.pt'
shutil.copy(best_path, f'/content/drive/MyDrive/{final_model_name}')

print(f"\n✅ SAVED: MyDrive/{final_model_name}")

model.export(format='onnx')
model.export(format='torchscript')

print("\n📦 Creating download package...")

download_folder = '/content/DOWNLOAD_PACKAGE'
os.makedirs(download_folder, exist_ok=True)

shutil.copy(best_path, f'{download_folder}/{final_model_name}')

training_folder = f'{drive_project_path}/yolov11n_pothole_distance_blur_v7'
result_files = ['results.png', 'confusion_matrix.png', 'F1_curve.png',
                'PR_curve.png', 'P_curve.png', 'R_curve.png', 'labels.jpg']

for file in result_files:
    src = f'{training_folder}/{file}'
    if os.path.exists(src):
        shutil.copy(src, f'{download_folder}/{file}')

for file in os.listdir('.'):
    if file.endswith(('.onnx', '.torchscript')):
        shutil.copy(file, f'{download_folder}/{file}')

with open(f'{download_folder}/RESULTS_SUMMARY.txt', 'w') as f:
    f.write("="*50 + "\n")
    f.write("DISTANCE + BLUR POTHOLE DETECTION\n")
    f.write("="*50 + "\n\n")
    f.write(f"Final Accuracy (mAP50): {final_accuracy:.1f}%\n")
    f.write(f"mAP50-95: {val_results.box.map:.4f}\n")
    f.write(f"Precision: {val_results.box.mp:.4f}\n")
    f.write(f"Recall: {val_results.box.mr:.4f}\n\n")
    f.write(f"Training Images: {train_images}\n")
    f.write(f"Validation Images: {val_images}\n")
    f.write(f"Optimized for: Distance + Blur Conditions\n")

shutil.make_archive('/content/Pothole_Distance_Blur_Complete', 'zip', download_folder)

print("\n📥 DOWNLOADING...")
files.download('/content/Pothole_Distance_Blur_Complete.zip')

print("\n✅ COMPLETE!")
print("="*50)
print("📂 Drive: MyDrive/pothole_distance_blur_training/")
print("📥 Downloaded: Pothole_Distance_Blur_Complete.zip")

Mounted at /content/drive
✅ Google Drive mounted!
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 67.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.8/89.8 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 52.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 83.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 116.9 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.12.0.88
    Uninstalling opencv-python-headless-4.12.0.88:
      Successfully uninstalled opencv-python-headless-4.12.0.88
  Attempting uninstall: idna
    Found existing installation: idna 3.10
    Uninstalling idna-3.10:
      Successfully uninstalled idna-3.10
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.conf


Extracting Dataset Version Zip to pothole-7 in yolov11:: 100%|██████████| 56136/56136 [00:08<00:00, 6730.06it/s]


Dataset downloaded to: /content/pothole-7

📁 Checking dataset structure...
Contents of dataset folder:
  - README.dataset.txt
  - train
  - data.yaml
  - test
  - valid
  - README.roboflow.txt

✅ Training images: 24348
✅ Validation images: 1941

Training YOLOv11n - Distance + Blur Pothole Detection...
Ultralytics 8.3.209 🚀 Python-3.12.11 torch-2.8.0+cu126 CUDA:0 (NVIDIA A100-SXM4-80GB, 81222MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=24, bgr=0.0, box=8.0, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/pothole-7/data.yaml, degrees=10, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=250, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=832, int8=False, iou=0.7, kera

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ COMPLETE!
📂 Drive: MyDrive/pothole_distance_blur_training/
📥 Downloaded: Pothole_Distance_Blur_Complete.zip
